In [ ]:
from urllib.parse import urljoin
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import re
import time
import pandas as pd
import requests

In [19]:
# Download PDFs
# ============================================================
# SETTINGS
# ============================================================

PDF_CATALOGUE_FILE = Path("data/ssc_pdf_catalogue.csv")
OUTPUT_DIR = Path("data/raw_pdfs/ssc")

BATCH_SIZE = 25

# Choose "all" for every batch, or an integer for a limited test.
BATCHES_TO_RUN = "all"
# BATCHES_TO_RUN = 2

REQUEST_DELAY_SECONDS = 1.5
TIMEOUT_SECONDS = 60
CHUNK_SIZE = 1024 * 128

# Stop rather than repeatedly hitting SSC if several files fail in sequence.
MAX_CONSECUTIVE_FAILURES = 5

HEADERS = {
    "User-Agent": "GovJobIntelligence/0.1 (educational project)",
    "Accept": "application/pdf,*/*",
}


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def safe_filename(filename):
    filename = str(filename or "ssc_document.pdf")
    filename = re.sub(r'[<>:"/\\|?*]', "_", filename)
    filename = re.sub(r"\s+", "_", filename).strip("._")

    if not filename.lower().endswith(".pdf"):
        filename += ".pdf"

    return filename


def sha256_of_file(file_path):
    digest = hashlib.sha256()

    with open(file_path, "rb") as file:
        for chunk in iter(lambda: file.read(CHUNK_SIZE), b""):
            digest.update(chunk)

    return digest.hexdigest()


def save_catalogue(catalogue):
    catalogue.to_csv(PDF_CATALOGUE_FILE, index=False, encoding="utf-8")


def mark_existing_file(catalogue, row_index, output_path):
    catalogue.loc[row_index, "local_pdf_path"] = str(output_path)
    catalogue.loc[row_index, "download_status"] = "already_exists"
    catalogue.loc[row_index, "actual_file_size_bytes"] = output_path.stat().st_size
    catalogue.loc[row_index, "pdf_sha256"] = sha256_of_file(output_path)
    catalogue.loc[row_index, "downloaded_at_utc"] = (
        datetime.now(timezone.utc).isoformat()
    )
    catalogue.loc[row_index, "download_error"] = ""


# ============================================================
# LOAD AND PREPARE CATALOGUE
# ============================================================

if not PDF_CATALOGUE_FILE.exists():
    raise FileNotFoundError(
        f"Catalogue not found: {PDF_CATALOGUE_FILE.resolve()}"
    )

catalogue_df = pd.read_csv(PDF_CATALOGUE_FILE)

if catalogue_df.empty:
    raise RuntimeError("The PDF catalogue is empty.")

catalogue_df = (
    catalogue_df
    .sort_values(
        by=["notice_created_at", "attachment_id"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

progress_columns = {
    "local_pdf_path": "",
    "download_status": "not_downloaded",
    "downloaded_at_utc": "",
    "pdf_sha256": "",
    "actual_file_size_bytes": pd.NA,
    "download_error": ""
}

for column_name, default_value in progress_columns.items():
    if column_name not in catalogue_df.columns:
        catalogue_df[column_name] = default_value

text_progress_columns = [
    "local_pdf_path",
    "download_status",
    "downloaded_at_utc",
    "pdf_sha256",
    "download_error"
]

for column_name in text_progress_columns:
    catalogue_df[column_name] = (
        catalogue_df[column_name]
        .fillna("")
        .astype("string")
    )

catalogue_df["actual_file_size_bytes"] = pd.to_numeric(
    catalogue_df["actual_file_size_bytes"],
    errors="coerce"
).astype("Int64")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

total_pdfs = len(catalogue_df)
total_batches = (total_pdfs + BATCH_SIZE - 1) // BATCH_SIZE

if BATCHES_TO_RUN == "all":
    batches_to_run = total_batches

elif isinstance(BATCHES_TO_RUN, int) and BATCHES_TO_RUN > 0:
    batches_to_run = min(BATCHES_TO_RUN, total_batches)

else:
    raise ValueError(
        "BATCHES_TO_RUN must be 'all' or a positive integer."
    )

print("=" * 65)
print("SSC AUTOMATED PDF DOWNLOAD")
print("=" * 65)
print(f"Total PDFs: {total_pdfs}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Total batches available: {total_batches}")
print(f"Batches running now: {batches_to_run}")
print(f"Output folder: {OUTPUT_DIR.resolve()}")

session = requests.Session()
session.headers.update(HEADERS)

consecutive_failures = 0
downloaded_this_run = 0
already_present_this_run = 0
failed_this_run = 0


# ============================================================
# AUTOMATIC BATCH LOOP
# ============================================================

for batch_number in range(1, batches_to_run + 1):
    start_index = (batch_number - 1) * BATCH_SIZE
    end_index = min(start_index + BATCH_SIZE, total_pdfs)

    print("\n" + "=" * 65)
    print(
        f"BATCH {batch_number}/{batches_to_run} "
        f"| PDFs {start_index + 1} to {end_index}"
    )
    print("=" * 65)

    for row_index in range(start_index, end_index):
        row = catalogue_df.iloc[row_index]

        notice_id = str(row["notice_id"])
        original_filename = safe_filename(row["original_filename"])

        filename = f"{notice_id}_{original_filename}"
        output_path = OUTPUT_DIR / filename
        temporary_path = OUTPUT_DIR / f"{filename}.part"

        print(f"\n[{row_index + 1}/{total_pdfs}] {original_filename}")

        try:
            # Previously downloaded files are retained and registered.
            if output_path.exists() and output_path.stat().st_size > 0:
                mark_existing_file(catalogue_df, row_index, output_path)

                already_present_this_run += 1
                consecutive_failures = 0

                print("Status: already exists")

            else:
                # Remove only an incomplete temporary file for this exact PDF.
                temporary_path.unlink(missing_ok=True)

                response = session.get(
                    row["pdf_url"],
                    stream=True,
                    timeout=TIMEOUT_SECONDS
                )
                response.raise_for_status()

                file_hash = hashlib.sha256()

                with open(temporary_path, "wb") as pdf_file:
                    for chunk in response.iter_content(chunk_size=CHUNK_SIZE):
                        if chunk:
                            pdf_file.write(chunk)
                            file_hash.update(chunk)

                with open(temporary_path, "rb") as check_file:
                    pdf_signature = check_file.read(4)

                if pdf_signature != b"%PDF":
                    temporary_path.unlink(missing_ok=True)

                    raise ValueError(
                        "SSC returned content that is not a valid PDF."
                    )

                temporary_path.replace(output_path)

                catalogue_df.loc[row_index, "local_pdf_path"] = str(output_path)
                catalogue_df.loc[row_index, "download_status"] = "downloaded"
                catalogue_df.loc[row_index, "actual_file_size_bytes"] = (
                    output_path.stat().st_size
                )
                catalogue_df.loc[row_index, "pdf_sha256"] = file_hash.hexdigest()
                catalogue_df.loc[row_index, "downloaded_at_utc"] = (
                    datetime.now(timezone.utc).isoformat()
                )
                catalogue_df.loc[row_index, "download_error"] = ""

                downloaded_this_run += 1
                consecutive_failures = 0

                print(
                    f"Status: downloaded "
                    f"({output_path.stat().st_size:,} bytes)"
                )

        except Exception as error:
            temporary_path.unlink(missing_ok=True)

            catalogue_df.loc[row_index, "download_status"] = "failed"
            catalogue_df.loc[row_index, "download_error"] = str(error)

            failed_this_run += 1
            consecutive_failures += 1

            print(f"Status: failed | {error}")

        # Persist progress after every PDF.
        save_catalogue(catalogue_df)

        if consecutive_failures >= MAX_CONSECUTIVE_FAILURES:
            print(
                f"\nStopped after {MAX_CONSECUTIVE_FAILURES} "
                "consecutive failures."
            )
            print("Run the same cell later to resume safely.")
            break

        time.sleep(REQUEST_DELAY_SECONDS)

    if consecutive_failures >= MAX_CONSECUTIVE_FAILURES:
        break


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 65)
print("DOWNLOAD RUN COMPLETE")
print("=" * 65)
print(f"Downloaded in this run: {downloaded_this_run}")
print(f"Already present: {already_present_this_run}")
print(f"Failed in this run: {failed_this_run}")

print("\nOverall catalogue status:")
display(
    catalogue_df["download_status"]
    .value_counts()
    .rename_axis("download_status")
    .reset_index(name="count")
)

print(f"\nUpdated catalogue:\n{PDF_CATALOGUE_FILE.resolve()}")

SSC AUTOMATED PDF DOWNLOAD
Total PDFs: 717
Batch size: 25
Total batches available: 29
Batches running now: 29
Output folder: /Users/aswathshakthi/Documents/My Files/Private/WorkSpace/PycharmProjects/Personal_Projects/16.TierTwoUK/data/raw_pdfs/ssc

BATCH 1/29 | PDFs 1 to 25

[1/717] writeup_29102026.pdf
Status: already exists

[2/717] Important_notice_revised_dated_26_10_2026.pdf
Status: already exists

[3/717] JE_2026_City_Live_Notice_23102026.pdf
Status: already exists

[4/717] tentative_vacancies_24102026.pdf
Status: already exists

[5/717] writeup_23102026.pdf
Status: already exists

[6/717] writeup_12102026.pdf
Status: already exists

[7/717] writeup_rhq.pdf
Status: already exists

[8/717] JE_Tentative_Vacancy_08102026.pdf
Status: already exists

[9/717] Final_Marks_Writeup_07102026.pdf
Status: already exists

[10/717] writeup_final_chsl_2025_07102026.pdf
Status: already exists

[11/717] writeup_07102026.pdf
Status: already exists

[12/717] advertisement_06102026.pdf
Status: alrea

,download_status,count
0,downloaded,691
1,already_exists,26



Updated catalogue:
/Users/aswathshakthi/Documents/My Files/Private/WorkSpace/PycharmProjects/Personal_Projects/16.TierTwoUK/data/ssc_pdf_catalogue.csv
